In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EmployeeDataPipeline") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Version:", spark.version)

In [ ]:
employees = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/employees.csv")
employees.show()
employees.printSchema()

In [ ]:
from pyspark.sql.functions import col

employees_transformed = employees.withColumn(
    "annual_salary",
    col("salary") * 12
)
employees_transformed.show()

In [ ]:
from pyspark.sql.functions import when

employees_transformed = employees_transformed.withColumn(
    "salary_category",
    when(col("salary") >= 70000, "High")
    .when(col("salary") >= 60000, "Medium")
    .otherwise("Low")
)
employees_transformed.show()

In [ ]:
employees_transformed.write \
    .mode("overwrite") \
    .parquet("output/parquet/employees")

In [ ]:
parquet_df = spark.read.parquet(
    "output/parquet/employees"
)

parquet_df.show()

In [ ]:
selected_employees = employees.select(
    "emp_id",
    "name",
    "department",
    "salary",
    "city"
)
selected_employees.show()

In [ ]:
high_salary_employees = selected_employees.filter(
    col("salary") > 60000
)
high_salary_employees.show()

In [ ]:
high_salary_employees.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/csv/high_salary_employees")

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Employee JDBC Demo")
    .config(
        "spark.jars",
        "./mssql-jdbc-13.2.0.jre11.jar"
    )
    .getOrCreate()
)

print(spark.version)

In [ ]:
jdbc_url = "jdbc:sqlserver://localhost:1433;databaseName=CompanyDB;encrypt=true;trustServerCertificate=true"

connection_properties = {
    "user": "sa",
    "password": "SqlServer@12345",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [ ]:
employees = spark.read \
    .jdbc(
        url=jdbc_url,
        table="employees",
        properties=connection_properties
    )
employees.show()

In [ ]:
from pyspark.sql.functions import col

employees_transformed = employees.withColumn(
    "annual_salary",
    col("salary") * 12
)
employees_transformed.show()

In [ ]:
high_salary = employees_transformed.filter(
    col("salary") >= 70000
)
high_salary.show()

In [9]:
high_salary.write \
    .mode("overwrite") \
    .parquet("output/parquet/sqlserver_employees")